In [18]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/hull-tactical-market-prediction/train.csv
/kaggle/input/hull-tactical-market-prediction/test.csv
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_inference_server.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/default_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/templates.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/base_gateway.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/relay.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/kaggle_evaluation.proto
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/__init__.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2.py
/kaggle/input/hull-tactical-market-prediction/kaggle_evaluation/core/generated/kaggle_evaluation_pb2_grpc.py
/kaggl

In [19]:
df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
TARGET_COL = 'market_forward_excess_returns'
df['target'] = df[TARGET_COL].shift(-1)
df_cleaned = df.dropna(axis=0).reset_index(drop=True)
df_cleaned

,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns,target
0,6969,0,0,0,0,0,-1,0,0,0,...,0.996693,0.534651,0.884921,-0.911387,0.979167,-0.847754,0.001145,0.000040,0.000797,0.004390
1,6970,0,0,0,0,0,0,0,1,0,...,0.997354,0.283256,0.769180,-0.842354,0.970238,-0.821913,0.004738,0.000040,0.004390,0.005669
2,6971,0,0,0,0,1,0,0,1,0,...,0.999339,0.713303,0.814153,-0.933089,0.951720,-0.817906,0.006016,0.000040,0.005669,0.001067
3,6972,0,0,0,0,1,0,1,1,0,...,1.000000,0.583019,0.809524,-1.051731,0.953042,-0.887412,0.001414,0.000039,0.001067,-0.007529
4,6973,0,0,0,0,1,0,0,0,1,...,0.999339,1.054972,0.567460,-1.138938,0.951058,-0.930502,-0.007182,0.000039,-0.007529,0.003067
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015,8984,0,0,0,0,0,0,1,0,1,...,0.839947,0.944593,0.715608,-0.692649,0.124669,-0.654045,-0.002896,0.000159,-0.003365,0.001990
2016,8985,0,0,0,0,0,0,0,0,0,...,0.837963,1.226772,0.822751,-0.707361,0.142857,-0.649616,0.002457,0.000155,0.001990,0.001845
2017,8986,0,0,0,0,0,0,0,0,0,...,0.837963,0.785877,0.805556,-0.715692,0.196098,-0.668289,0.002312,0.000156,0.001845,0.002424
2018,8987,0,0,1,0,0,0,0,0,0,...,0.787698,0.834898,0.823413,-0.723949,0.133929,-0.670946,0.002891,0.000156,0.002424,0.007843


In [20]:
import warnings

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.options.mode.chained_assignment = None 

df = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
TARGET_COL = 'market_forward_excess_returns'
df['target'] = df[TARGET_COL].shift(-1) 

W_train = 1000
start_test_index = W_train

# Corrected Loop Variable Definitions
all_predictions = []
all_true_returns = []

print(f"Total available rows after cleaning: {len(df_cleaned)}")
print(f"Starting WFW at index: {start_test_index}")

Total available rows after cleaning: 2020
Starting WFW at index: 1000


In [21]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' 

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
# Check if TensorFlow/Keras is available (required for LSTM)
try:
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from tensorflow.keras.optimizers import Adam
    LSTM_AVAILABLE = True
except ImportError:
    print("TensorFlow/Keras not found. Running only XGBoost and RandomForest.")
    LSTM_AVAILABLE = False



xgb_model = XGBRegressor(n_estimators=50, max_depth=3, random_state=42, n_jobs=-1)
rf_model = RandomForestRegressor(n_estimators=50, max_depth=3, random_state=42, n_jobs=-1)

def build_lstm_model(input_shape):
    model = Sequential([
        LSTM(units=32, return_sequences=False, input_shape=input_shape),
        Dense(units=1)
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

def prep_lstm_data(X_data, Y_data, lookback=5):
    # Scale ALL features for LSTM
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_scaled = scaler.fit_transform(X_data)
    
    X_3d = np.array([X_scaled[i-lookback:i] for i in range(lookback, len(X_scaled))])
    Y_aligned = Y_data.iloc[lookback:].values
    
    return X_3d, Y_aligned, scaler

In [ ]:
# --- Define Weights and Lookback ---
W_xgb, W_rf, W_lstm = 0.4, 0.4, 0.2
LSTM_LOOKBACK = 5
DROP_COLS = ['date_id', TARGET_COL, 'target', 'risk_free_rate']

# --- Prepare Initial Training Data for LSTM ---
if LSTM_AVAILABLE:
    # Use the first W_train rows for training LSTM
    X_init = df_cleaned.iloc[:W_train].drop(columns=DROP_COLS, errors='ignore')
    Y_init = df_cleaned.iloc[:W_train]['target']

    # Scale + reshape data for LSTM
    def prep_lstm_data(X_data, Y_data, lookback=5):
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_scaled = scaler.fit_transform(X_data)
        X_3d = np.array([X_scaled[i-lookback:i] for i in range(lookback, len(X_scaled))])
        Y_aligned = Y_data.iloc[lookback:].values
        return X_3d, Y_aligned, scaler

    X_lstm_train, Y_lstm_train, scaler = prep_lstm_data(X_init, Y_init, lookback=LSTM_LOOKBACK)

    # Build and train LSTM once
    from tensorflow.keras import Input
    def build_lstm_model(input_shape):
        model = Sequential([
            Input(shape=input_shape),
            LSTM(units=32, return_sequences=False),
            Dense(units=1)
        ])
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
        return model

    lstm_model = build_lstm_model((X_lstm_train.shape[1], X_lstm_train.shape[2]))
    lstm_model.fit(X_lstm_train, Y_lstm_train, epochs=10, batch_size=16, verbose=0)

print("Initial LSTM training complete.")

# --- Walk-Forward Validation Loop ---
for i in range(start_test_index, len(df_cleaned)):
    train_data = df_cleaned.iloc[i - W_train : i]
    test_data = df_cleaned.iloc[i : i + 1]

    X_train = train_data.drop(columns=DROP_COLS, errors='ignore')
    Y_train = train_data['target']
    X_test = test_data.drop(columns=DROP_COLS, errors='ignore')

    # --- A. Train/Predict XGBoost & Random Forest ---
    xgb_model.fit(X_train, Y_train)
    rf_model.fit(X_train, Y_train)

    pred_xgb = xgb_model.predict(X_test)[0]
    pred_rf  = rf_model.predict(X_test)[0]

    # --- B. Predict with Pre-Trained LSTM ---
    pred_lstm = 0.0
    if LSTM_AVAILABLE and i > LSTM_LOOKBACK:
        X_test_lstm_input = df_cleaned.iloc[i - LSTM_LOOKBACK : i].drop(columns=DROP_COLS, errors='ignore')
        X_test_scaled = scaler.transform(X_test_lstm_input)
        X_test_3d = X_test_scaled[np.newaxis, :, :]  # shape = (1, lookback, n_features)

        pred_lstm = lstm_model.predict(X_test_3d, verbose=0)[0][0]

    # --- C. Ensemble Prediction ---
    ensemble_prediction = (W_xgb * pred_xgb) + (W_rf * pred_rf) + (W_lstm * pred_lstm)

    # Store results
    all_predictions.append(ensemble_prediction)
    all_true_returns.append(test_data['target'].values[0])

    if i % 500 == 0:
        print(f"Completed day {i} / {len(df_cleaned)}")

print("Walk-Forward Validation Complete.")


Initial LSTM training complete.
Completed day 1000 / 2020
Completed day 1500 / 2020


In [ ]:
import matplotlib.pyplot as plt

# Convert to numpy arrays
preds = np.array(all_predictions)
true_vals = np.array(all_true_returns)

# Cumulative returns (assuming predictions are daily returns)
cumulative_true = np.cumsum(true_vals)
cumulative_pred = np.cumsum(preds)

plt.figure(figsize=(12,6))
plt.plot(cumulative_true, label="True Cumulative Returns")
plt.plot(cumulative_pred, label="Predicted Cumulative Returns")
plt.legend()
plt.title("Model Prediction vs True Market Returns")
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(all_true_returns, all_predictions)
r2 = r2_score(all_true_returns, all_predictions)

print(f"Mean Squared Error: {mse:.6f}")
print(f"R^2 Score: {r2:.6f}")